<div style="border-bottom: 3px solid #6495ED;">
    <h1 style="color:#6495ED;">Clustering</h1>
</div>

## Data Reading

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("CSVTables")

df_customer = pd.read_csv(DATA_DIR / "DF_CUSTOMER_FINAL.csv", delimiter=";")

#print(df_customer.shape)
df_customer

In [ ]:
id_col = "customer_key"

drop_cols = [id_col, "rfm_score", "favorite_brand", "favorite_hs"]  # strings/derivados
candidate_cols = [c for c in df_customer.columns if c not in drop_cols]

df_cluster = df_customer[candidate_cols].copy()
df_cluster = df_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)
df_cluster


# Pre-Processing

## Skeweness

In [ ]:
import numpy as np

# df_cluster: já sem strings/derivados, só numéricas
X = df_cluster.copy()

# garantir numéricos
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

# calcular skewness (Fisher-Pearson) e % zeros (útil para contagens)
skew = X.skew(numeric_only=True)
pct_zero = (X == 0).mean()

skew_df = pd.DataFrame({
    "skewness": skew,
    "pct_zero": pct_zero,
    "min": X.min(),
    "max": X.max()
}).sort_values("skewness", ascending=False)

print(skew_df.head(20))

## Log1p

In [ ]:
import numpy as np

# df_cluster: só numéricas (já sem strings/derivados)
X = df_cluster.copy().apply(pd.to_numeric, errors="coerce").fillna(0)

# 1) Corrigir violação de domínio (apenas 1 caso)
if "discount_sensitivity" in X.columns:
    neg_mask = X["discount_sensitivity"] < 0
    n_neg = int(neg_mask.sum())
    if n_neg > 0:
        print(f"[WARN] discount_sensitivity tem {n_neg} valores negativos. A fazer clip para 0.")
        X.loc[neg_mask, "discount_sensitivity"] = 0.0

# 2) Calcular skewness
skew = X.skew(numeric_only=True)
pct_zero = (X == 0).mean()
mins = X.min(numeric_only=True)   
maxs = X.max(numeric_only=True)

skew_df = pd.DataFrame({
    "skewness": skew,
    "pct_zero": pct_zero,
    "min": mins,
    "max": maxs
}).sort_values("skewness", ascending=False)

print(skew_df.head(20))

SKEW_THR = 1.0
exclude = {"return_rate", "discount_sensitivity"}  # rates: não aplicar log por defeito

log_cols = [
    c for c in X.columns
    if (c not in exclude)
    and (skew.get(c, 0) > SKEW_THR)
    and (mins.get(c, 0) >= 0)
]

print("Columns selected for log1p:", log_cols)

# 3) Aplicar log1p com segurança
df_log = X.copy()
for c in log_cols:
    df_log[c] = np.log1p(df_log[c])

# 4) verificar melhoria de skewness
skew_after = df_log.skew(numeric_only=True)
check = (
    pd.DataFrame({"skew_before": skew, "skew_after": skew_after})
    .loc[log_cols]
    .sort_values("skew_before", ascending=False)
)
print(check.head(15))


## Winsorize

In [ ]:
#Winsorize

df_model = df_log.copy()

# cap p1-p99 para estabilizar
for c in df_model.columns:
    lo, hi = df_model[c].quantile(0.01), df_model[c].quantile(0.99)
    df_model[c] = df_model[c].clip(lo, hi)

print("df_model shape:", df_model.shape)

## Correlation

In [ ]:
pear = df_model.corr(method="pearson")
spear = df_model.corr(method="spearman")

thr = 0.8

pairs = []
cols = pear.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        a, b = cols[i], cols[j]
        p = pear.loc[a, b]
        s = spear.loc[a, b]
        if abs(p) >= thr and abs(s) >= thr:
            pairs.append((a, b, p, s))

pairs_df = pd.DataFrame(pairs, columns=["feat_a", "feat_b", "pearson", "spearman"])\
            .sort_values(by=["pearson", "spearman"], key=lambda x: x.abs(), ascending=False)

print("Highly correlated pairs (|Pearson| & |Spearman| >= 0.8):", len(pairs_df))
print(pairs_df.head(20))

In [ ]:
#Analisámos os dados e decidimos tirar estas colunas, ver notas 

drop_cols = [
    "average_order_value",
    "favorite_hs_share",
    "favorite_hs_count",
    "favorite_brand_count",
    "total_paid_item_lines",
]

df_model_v1 = df_model.drop(columns=[c for c in drop_cols if c in df_model.columns], errors="ignore")
print("df_model_v1 shape:", df_model_v1.shape)

In [ ]:
pear_v1 = df_model_v1.corr(method="pearson")
spear_v1 = df_model_v1.corr(method="spearman")

thr = 0.8
pairs = []
cols = pear_v1.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        a, b = cols[i], cols[j]
        p, s = pear_v1.loc[a,b], spear_v1.loc[a,b]
        if abs(p) >= thr and abs(s) >= thr:
            pairs.append((a,b,p,s))

pairs_v1 = pd.DataFrame(pairs, columns=["feat_a","feat_b","pearson","spearman"]) \
            .sort_values(by=["pearson","spearman"], key=lambda x: x.abs(), ascending=False)

print("Remaining highly correlated pairs:", len(pairs_v1))
print(pairs_v1)


In [ ]:
# correlação recency vs tenure por grupos de frequency
tmp = pd.concat([df_customer[["frequency"]], df_model[["recency","tenure_days"]]], axis=1).dropna()
tmp["freq_group"] = np.where(tmp["frequency"] <= 1, "freq=1", "freq>=2")

for g, sub in tmp.groupby("freq_group"):
    print(g, "pearson:", sub["recency"].corr(sub["tenure_days"], method="pearson"),
              "spearman:", sub["recency"].corr(sub["tenure_days"], method="spearman"))


In [ ]:
drop_more = ["avg_interpurchase_days", "std_order_value"]
df_model_final = df_model_v1.drop(columns=[c for c in drop_more if c in df_model_v1.columns], errors="ignore")
print("df_model_final shape:", df_model_final.shape)

In [ ]:
pear_f = df_model_final.corr(method="pearson")
spear_f = df_model_final.corr(method="spearman")

thr = 0.8
pairs = []
cols = pear_f.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        a,b = cols[i], cols[j]
        p,s = pear_f.loc[a,b], spear_f.loc[a,b]
        if abs(p) >= thr and abs(s) >= thr:
            pairs.append((a,b,p,s))

pairs_final = pd.DataFrame(pairs, columns=["feat_a","feat_b","pearson","spearman"])
print("Remaining highly correlated pairs:", len(pairs_final))
print(pairs_final)


## Scaling (2 methods)

In [ ]:
from sklearn.preprocessing import RobustScaler

# df_model_final: só features numéricas finais, já com log1p + cap aplicado
scaler = RobustScaler()

df_scaled = scaler.fit_transform(df_model_final)

# guardar num DataFrame para inspeção
df_scaled_rb = pd.DataFrame(df_scaled, columns=df_model_final.columns, index=df_model_final.index)

print("X_scaled shape:", df_scaled_rb.shape)
print("\nScaled summary (mean/std - não precisam ser 0/1 com RobustScaler):")
print(df_scaled_rb.describe().loc[["mean", "std", "min", "max"]])

In [ ]:
#comparar com standardScaler

from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
df_std = std_scaler.fit_transform(df_model_final)

df_scaled_std = pd.DataFrame(df_std, columns=df_model_final.columns, index=df_model_final.index)

print("df_std shape:", df_scaled_std.shape)
print("\nStandardScaler summary (mean≈0, std≈1):")
print(df_scaled_std.describe().loc[["mean","std","min","max"]])


# Clustering

## Clusters Generation 

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

def evaluate_kmeans(X, k_values=range(2, 11), seeds=(0,1,2,3,4), name="X"):
    """
    X: np.array ou pd.DataFrame (já escalado)
    k_values: ks a testar
    seeds: seeds para estabilidade
    """
    if isinstance(X, pd.DataFrame):
        X_mat = X.values
    else:
        X_mat = np.asarray(X)

    results = []

    for k in k_values:
        silhouettes = []
        inertias = []
        labels_list = []
        min_cluster_fracs = []

        for seed in seeds:
            km = KMeans(
                n_clusters=k,
                init="k-means++",
                n_init=10,
                max_iter=300,
                random_state=seed
            )
            labels = km.fit_predict(X_mat)

            # silhouette (só faz sentido se houver >1 cluster e sem clusters vazios)
            sil = silhouette_score(X_mat, labels, sample_size=10000, random_state=seed)
            silhouettes.append(sil)

            inertias.append(km.inertia_)
            labels_list.append(labels)

            # tamanho mínimo de cluster (fração)
            counts = np.bincount(labels)
            min_cluster_fracs.append(counts.min() / len(labels))

        # estabilidade ARI média entre pares de runs
        aris = []
        for i in range(len(labels_list)):
            for j in range(i+1, len(labels_list)):
                aris.append(adjusted_rand_score(labels_list[i], labels_list[j]))
        ari_mean = float(np.mean(aris)) if aris else np.nan

        results.append({
            "scaler": name,
            "k": k,
            "sil_mean": float(np.mean(silhouettes)),
            "sil_std": float(np.std(silhouettes)),
            "inertia_mean": float(np.mean(inertias)),
            "ari_mean": ari_mean,
            "min_cluster_frac_mean": float(np.mean(min_cluster_fracs)),
        })

    return pd.DataFrame(results).sort_values(["k"])

In [ ]:
k_values = range(2, 11)
seeds = (0, 1, 2)

res_rb  = evaluate_kmeans(df_scaled_rb,  k_values=k_values, seeds=seeds, name="RobustScaler")
res_std = evaluate_kmeans(df_scaled_std, k_values=k_values, seeds=seeds, name="StandardScaler")

results = pd.concat([res_rb, res_std], ignore_index=True)

# ver resultados
print(results)

In [ ]:
# Ligação à df_customer

def ensure_df(X, index, columns=None, name="X"):
    """Garante que X é DataFrame com index igual ao df_customer.index."""
    if isinstance(X, pd.DataFrame):
        # reindex para garantir ordem
        X = X.reindex(index)
        return X
    X = np.asarray(X)
    if columns is None:
        columns = [f"f{i}" for i in range(X.shape[1])]
    return pd.DataFrame(X, index=index, columns=columns)

# assegurar que os índices batem certo
df_scaled_rb  = ensure_df(df_scaled_rb,  df_customer.index, name="df_scaled_rb")
df_scaled_std = ensure_df(df_scaled_std, df_customer.index, name="df_scaled_std")

print("Aligned shapes:")
print("df_customer:", df_customer.shape)
print("df_scaled_rb:", df_scaled_rb.shape)
print("df_scaled_std:", df_scaled_std.shape)

In [ ]:
# Treinar modelos candidatos
def fit_kmeans_and_label(X_df, k, seed=42, n_init=20):
    km = KMeans(n_clusters=k, init="k-means++", n_init=n_init, max_iter=300, random_state=seed)
    labels = km.fit_predict(X_df.values)
    return km, labels

# Candidato A: RobustScaler + K=4
k_rb = 4
km_rb4, labels_rb4 = fit_kmeans_and_label(df_scaled_rb, k_rb, seed=42, n_init=20)
df_customer["cluster_rb4"] = labels_rb4

# Candidato B: StandardScaler + K=6 (muda para 7 se quiseres)
k_std = 6
km_std6, labels_std6 = fit_kmeans_and_label(df_scaled_std, k_std, seed=42, n_init=20)
df_customer["cluster_std6"] = labels_std6

print("\nCluster sizes (RB4):")
print(df_customer["cluster_rb4"].value_counts(normalize=True).sort_index().round(4))
print("\nCluster sizes (STD6):")
print(df_customer["cluster_std6"].value_counts(normalize=True).sort_index().round(4))

# (Opcional, rápido) silhouette por amostragem, só para sanity check
def quick_silhouette(X_df, labels, sample_size=10000, seed=42):
    n = len(labels)
    ss = min(sample_size, n)
    return silhouette_score(X_df.values, labels, sample_size=ss, random_state=seed)

print("\nSilhouette(sample) RB4:", round(quick_silhouette(df_scaled_rb, labels_rb4), 4))
print("Silhouette(sample) STD6:", round(quick_silhouette(df_scaled_std, labels_std6), 4))

## Profiling

In [ ]:
def cluster_profile(df, cluster_col, numeric_cols, top_cats=None, top_n=5):
    """
    df: df_customer
    cluster_col: coluna com labels
    numeric_cols: colunas numéricas para perfil (mediana e média)
    top_cats: lista de colunas categóricas para top values (ex: ["favorite_brand","favorite_hs"])
    """
    out = {}

    # Tamanhos
    sizes = df[cluster_col].value_counts().sort_index()
    sizes_pct = (sizes / len(df)).round(4)
    out["sizes"] = pd.DataFrame({"n_customers": sizes, "pct_customers": sizes_pct})

    # Perfil numérico
    prof_med = df.groupby(cluster_col)[numeric_cols].median(numeric_only=True)
    prof_mean = df.groupby(cluster_col)[numeric_cols].mean(numeric_only=True)
    out["numeric_median"] = prof_med
    out["numeric_mean"] = prof_mean

    # Top categóricos
    if top_cats:
        tops = {}
        for c in top_cats:
            if c not in df.columns:
                continue
            tops[c] = (
                df.groupby(cluster_col)[c]
                .apply(lambda s: s.dropna().astype(str).value_counts().head(top_n))
            )
        out["top_cats"] = tops

    return out

# Colunas numéricas: ajusta se necessário (mantém coerência com features do clustering + úteis p/ persona)
numeric_cols = [
    "recency", "frequency", "monetary", "tenure_days",
    "discount_sensitivity", "return_rate",
    "unique_brands", "unique_hs",
    "avg_items_per_order",
    "brand_loyalty", "favorite_brand_loyalty",
]

# garante que só usamos colunas existentes
numeric_cols = [c for c in numeric_cols if c in df_customer.columns]

top_cat_cols = []
for c in ["favorite_brand", "favorite_hs"]:
    if c in df_customer.columns:
        top_cat_cols.append(c)

# Perfis
profile_rb4 = cluster_profile(df_customer, "cluster_rb4", numeric_cols, top_cats=top_cat_cols, top_n=5)
profile_std6 = cluster_profile(df_customer, "cluster_std6", numeric_cols, top_cats=top_cat_cols, top_n=5)

print("\n=== RB4 sizes ===")
print(profile_rb4["sizes"])

print("\n=== RB4 numeric medians (head) ===")
print(profile_rb4["numeric_median"].round(3).head())

if "top_cats" in profile_rb4 and profile_rb4["top_cats"]:
    for col, series in profile_rb4["top_cats"].items():
        print(f"\n=== RB4 top {col} per cluster ===")
        print(series)

print("\n=== STD6 sizes ===")
print(profile_std6["sizes"])

print("\n=== STD6 numeric medians (head) ===")
print(profile_std6["numeric_median"].round(3).head())

if "top_cats" in profile_std6 and profile_std6["top_cats"]:
    for col, series in profile_std6["top_cats"].items():
        print(f"\n=== STD6 top {col} per cluster ===")
        print(series)


In [ ]:
#Diagnóstico do cluster pequeno de std
CLUSTER_COL = "cluster_std6"
SMALL_CLUSTER = 5

# 1) Separar cluster pequeno vs resto
df_small = df_customer[df_customer[CLUSTER_COL] == SMALL_CLUSTER].copy()
df_rest  = df_customer[df_customer[CLUSTER_COL] != SMALL_CLUSTER].copy()

print("Small cluster size:", df_small.shape[0], "| pct:", round(df_small.shape[0]/df_customer.shape[0], 4))
print("Rest size:", df_rest.shape[0])

# 2) Features numéricas a comparar (ajusta se necessário)
num_cols = [
    "recency","tenure_days","frequency","monetary",
    "discount_sensitivity","return_rate",
    "unique_brands","unique_hs","avg_items_per_order",
    "brand_loyalty","favorite_brand_loyalty",
]
num_cols = [c for c in num_cols if c in df_customer.columns]

# 3) Summary estatístico: pequeno cluster
print("\n=== SMALL CLUSTER describe (selected cols) ===")
display(df_small[num_cols].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).T)

# 4) Summary estatístico: resto
print("\n=== REST describe (selected cols) ===")
display(df_rest[num_cols].describe(percentiles=[.1,.25,.5,.75,.9,.95,.99]).T)

# 5) Tabela comparativa (medianas + rácios)
small_med = df_small[num_cols].median(numeric_only=True)
rest_med  = df_rest[num_cols].median(numeric_only=True)

compare = pd.DataFrame({
    "median_small": small_med,
    "median_rest": rest_med,
    "ratio_small_vs_rest": (small_med / rest_med.replace(0, np.nan))
}).sort_values("ratio_small_vs_rest", ascending=False)

print("\n=== MEDIAN comparison (sorted by ratio) ===")
display(compare)

# 6) Percentagens úteis (ex: quantos têm frequency>=2, monetary alto, etc.)
def pct(cond):
    return float(cond.mean()) if len(cond) else np.nan

checks = {
    "pct_freq_ge_2_small": pct(df_small["frequency"] >= 2) if "frequency" in df_small.columns else np.nan,
    "pct_freq_ge_2_rest":  pct(df_rest["frequency"] >= 2) if "frequency" in df_rest.columns else np.nan,
    "pct_monetary_ge_p95_small": np.nan,
    "pct_monetary_ge_p95_rest":  np.nan,
}

if "monetary" in df_customer.columns:
    p95 = df_customer["monetary"].quantile(0.95)
    checks["pct_monetary_ge_p95_small"] = pct(df_small["monetary"] >= p95)
    checks["pct_monetary_ge_p95_rest"]  = pct(df_rest["monetary"] >= p95)

print("\n=== Quick checks ===")
for k,v in checks.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) and not np.isnan(v) else f"{k}: {v}")

# 7) (Opcional) Top brands/HS do cluster pequeno — bom para interpretar
for cat_col in ["favorite_brand", "favorite_hs"]:
    if cat_col in df_customer.columns:
        print(f"\n=== Top {cat_col} in SMALL cluster ===")
        display(df_small[cat_col].astype(str).value_counts().head(10))

In [ ]:
#Conlui-se que o cluster 6 não ajuda a construir personas de fidelização/recommender, é um cluster “refund-only”. Seguimos com rb

CL = "cluster_rb4"
key_cols = ["recency","monetary","discount_sensitivity","avg_items_per_order","unique_brands","unique_hs","tenure_days","frequency"]
key_cols = [c for c in key_cols if c in df_customer.columns]

global_med = df_customer[key_cols].median(numeric_only=True)

# percentis por cluster
q = df_customer.groupby(CL)[key_cols].quantile([0.25,0.5,0.75]).unstack()
q.columns = [f"{col}_p{int(p*100)}" for col,p in q.columns]
sizes = df_customer[CL].value_counts(normalize=True).sort_index().rename("pct")

persona_table = pd.concat([sizes, q], axis=1)

# deltas (mediana do cluster / mediana global)
cluster_meds = df_customer.groupby(CL)[key_cols].median(numeric_only=True)
deltas = (cluster_meds / global_med).add_prefix("ratio_to_global_")

persona_table = persona_table.merge(deltas, left_index=True, right_index=True)

print(persona_table.round(3))

## Clusters Visualization

In [ ]:
import matplotlib.pyplot as plt

CL = "cluster_rb4"

# features mais úteis para explicar personas
feat_core = ["recency", "monetary", "discount_sensitivity", "avg_items_per_order", "unique_brands", "unique_hs"]
feat_core = [c for c in feat_core if c in df_customer.columns]

# garantir cluster como int (ordem consistente)
df_customer[CL] = pd.to_numeric(df_customer[CL], errors="coerce").astype("Int64")

In [ ]:
#Tamanho dos clusters

sizes = df_customer[CL].value_counts().sort_index()
pct = (sizes / sizes.sum()) * 100

plt.figure()
plt.bar(pct.index.astype(str), pct.values)
plt.title("Distribuição de clientes por cluster (RB4)")
plt.xlabel("Cluster")
plt.ylabel("% de clientes")
for i, v in enumerate(pct.values):
    plt.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
plt.show()

In [ ]:
#Heat-map de medianas normalizadas 

med_global = df_customer[feat_core].median(numeric_only=True)
med_cluster = df_customer.groupby(CL)[feat_core].median(numeric_only=True)

ratio = med_cluster / med_global  # fingerprint

plt.figure(figsize=(10, 3.5))
plt.imshow(ratio.values, aspect="auto")
plt.xticks(range(len(feat_core)), feat_core, rotation=45, ha="right")
plt.yticks(range(ratio.shape[0]), ratio.index.astype(str))
plt.title("Fingerprint das personas (mediana do cluster / mediana global)")
plt.colorbar(label="ratio")
plt.tight_layout()
plt.show()

print(ratio.round(3))

In [ ]:
#Boxplot para 3 key features 

def boxplot_by_cluster(col):
    data = [df_customer[df_customer[CL] == k][col].dropna().values for k in sorted(df_customer[CL].dropna().unique())]
    labels = [str(k) for k in sorted(df_customer[CL].dropna().unique())]

    plt.figure()
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.title(f"{col} por cluster (RB4) – boxplot (sem outliers extremos)")
    plt.xlabel("Cluster")
    plt.ylabel(col)
    plt.show()

for col in ["recency", "monetary", "discount_sensitivity"]:
    if col in df_customer.columns:
        boxplot_by_cluster(col)

In [ ]:
#Monetary vs Recency

plot_df = df_customer[[CL, "recency", "monetary"]].dropna().copy()

# amostrar para visual (evita overplotting)
plot_df = plot_df.sample(n=min(15000, len(plot_df)), random_state=42)

plt.figure()
for k in sorted(plot_df[CL].unique()):
    sub = plot_df[plot_df[CL] == k]
    plt.scatter(sub["recency"], sub["monetary"], s=8, alpha=0.4, label=f"Cluster {k}")

plt.title("Monetary vs Recency (amostra) – clusters RB4")
plt.xlabel("Recency (dias desde última compra)")
plt.ylabel("Monetary (total gasto)")
plt.legend()
plt.show()

In [ ]:
CL = "cluster_rb4"
hs_col = "favorite_hs"
top_n = 3

# sanity check
assert CL in df_customer.columns, f"Falta {CL}"
assert hs_col in df_customer.columns, f"Falta {hs_col}"

tmp = df_customer[[CL, hs_col]].dropna().copy()
tmp[CL] = pd.to_numeric(tmp[CL], errors="coerce").astype("Int64")
tmp[hs_col] = tmp[hs_col].astype(str)

# contar ocorrências por (cluster, hs)
counts = (
    tmp.groupby([CL, hs_col])
    .size()
    .rename("n")
    .reset_index()
)

# percentagem dentro de cada cluster
counts["pct"] = counts["n"] / counts.groupby(CL)["n"].transform("sum")

# top N HS por cluster
top_hs = (
    counts.sort_values([CL, "pct"], ascending=[True, False])
    .groupby(CL)
    .head(top_n)
)

pivot = top_hs.pivot(index=CL, columns=hs_col, values="pct").fillna(0).sort_index()

# plot stacked bar
plt.figure()
bottom = np.zeros(len(pivot))
x = np.arange(len(pivot.index))

for hs in pivot.columns:
    vals = pivot[hs].values
    plt.bar(x, vals, bottom=bottom, label=hs)
    bottom += vals

plt.xticks(x, pivot.index.astype(str))
plt.title(f"Top {top_n} HS por cluster (proporção dentro do cluster)")
plt.xlabel("Cluster")
plt.ylabel("Proporção")
plt.legend(title="HS")
plt.show()

print(pivot.round(3))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import parallel_coordinates

CL = "cluster_rb4"

feat = [
    "recency","frequency","monetary","tenure_days",
    "discount_sensitivity","avg_items_per_order",
    "unique_brands","unique_hs","brand_loyalty","favorite_brand_loyalty"
]
feat = [c for c in feat if c in df_customer.columns]

# 1) medianas por cluster
med = df_customer.groupby(CL)[feat].median(numeric_only=True)

# 2) robust z-score global: (med_cluster - med_global) / IQR_global
global_med = df_customer[feat].median(numeric_only=True)
q75 = df_customer[feat].quantile(0.75, numeric_only=True)
q25 = df_customer[feat].quantile(0.25, numeric_only=True)
iqr = (q75 - q25).replace(0, np.nan)

med_z = (med - global_med) / iqr
med_z = med_z.replace([np.inf, -np.inf], np.nan).fillna(0)

# 3) preparar df para parallel coordinates e forçar ordem 0,1,2,3
pc_df = med_z.reset_index()
pc_df[CL] = pd.to_numeric(pc_df[CL], errors="coerce").astype(int)
pc_df = pc_df.sort_values(CL).copy()
pc_df[CL] = pc_df[CL].astype(str)

# 4) cores (na mesma ordem dos clusters após sort)
colors = ["#0B1F8A", "#8B0000", "#006400", "#4B2E1F"]  # azul, vermelho, verde, castanho escuros

fig, ax = plt.subplots(figsize=(14, 6))
parallel_coordinates(pc_df, class_column=CL, cols=feat, color=colors, alpha=0.95, ax=ax)

ax.axhline(0, linewidth=1)
ax.set_title("Parallel Coordinates — fingerprint dos clusters (mediana robust-z vs global)")
ax.set_ylabel("Robust z-score (mediana_cluster vs global, escalado por IQR)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")

# tirar grid
ax.grid(False)

# (opcional) remover as bordas de cima e direita para ficar mais clean
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# Criar um Radar Chart para todos os clusters usando as medianas robust-z score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import pi
CL = "cluster_rb4"
feat = [
    "recency","frequency","monetary","tenure_days",
    "discount_sensitivity","avg_items_per_order"
]
feat = [c for c in feat if c in df_customer.columns]
# 1) medianas por cluster
med = df_customer.groupby(CL)[feat].median(numeric_only=True)
# 2) robust z-score global: (med_cluster - med_global) / IQR_global
global_med = df_customer[feat].median(numeric_only=True)
q75 = df_customer[feat].quantile(0.75, numeric_only=True)
q25 = df_customer[feat].quantile(0.25, numeric_only=True)
iqr = (q75 - q25).replace(0, np.nan)

med_z = (med - global_med) / iqr
med_z = med_z.replace([np.inf, -np.inf], np.nan).fillna(0)
# 3) preparar dados para radar chart
categories = feat
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # fechar o círculo
# 4) plotar radar chart
plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)
colors = ["b", "r", "g", "m"]  # cores para os clusters
for idx, row in med_z.iterrows():
    values = row.tolist()
    values += values[:1]  # fechar o círculo
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=f'Cluster {idx}')
    ax.fill(angles, values, alpha=0.25)
# 5) customizar o gráfico
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_rlabel_position(30)
plt.yticks([-2, -1, 0, 1, 2], ["-2", "-1", "0", "1", "2"], color="grey", size=8)
plt.ylim(-3, 3)
plt.title("Radar Chart — Clusters Fingerprint  (robust-z median vs global)", size=15, y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
#mudar cor do fundo do gráfico
ax.set_facecolor('#fff7eb')
#mudar a cor do fundo da figura
plt.gcf().set_facecolor('#fff7eb')
plt.show()

## Personas Extraction

In [ ]:
CL = "cluster_rb4"
assert CL in df_customer.columns, f"Falta a coluna {CL} em df_customer."

persona_map = {
    0: "Deal Hunters (Promo-driven)",
    1: "Dormant (Churn risk)",
    2: "Core Actives (Low-basket)",
    3: "High-Value Explorers"
}

df_customer[CL] = pd.to_numeric(df_customer[CL], errors="coerce").astype("Int64")
df_customer["persona"] = df_customer[CL].map(persona_map)

print("Missing personas:", df_customer["persona"].isna().sum())
print(df_customer[[CL, "persona"]].dropna().head(8))

In [ ]:
feat_num = [
    "recency", "frequency", "monetary", "tenure_days",
    "discount_sensitivity", "return_rate",
    "avg_items_per_order", "unique_brands", "unique_hs"
]
feat_num = [c for c in feat_num if c in df_customer.columns]

# percentagens
sizes = (
    df_customer.groupby(CL)
    .size()
    .rename("n_customers")
    .to_frame()
)
sizes["pct"] = (sizes["n_customers"] / sizes["n_customers"].sum()).round(3)

# quantis por cluster
def q25(x): return x.quantile(0.25)
def q50(x): return x.quantile(0.50)
def q75(x): return x.quantile(0.75)

q = (
    df_customer.groupby(CL)[feat_num]
    .agg([q25, q50, q75])
)

# achatar nomes
q.columns = [f"{col}_{stat}" for col, stat in q.columns]

# medianas globais (para ratios)
global_median = df_customer[feat_num].median(numeric_only=True)

# ratios (mediana do cluster / mediana global)
for f in feat_num:
    q[f"ratio_to_global_{f}"] = (q[f"{f}_q50"] / global_median[f]).replace([np.inf, -np.inf], np.nan)

persona_table = sizes.join(q)
persona_table = persona_table.sort_values("pct", ascending=False)

# adicionar nome da persona
persona_table["persona"] = persona_table.index.map(persona_map)

print(persona_table[["persona","pct"] + [c for c in persona_table.columns if "ratio_to_global" in c]].head(10))


In [ ]:
def top_values_per_cluster(df, cluster_col, value_col, top_n=5):
    if value_col not in df.columns:
        print(f"[SKIP] {value_col} não existe.")
        return None
    tmp = df[[cluster_col, value_col]].dropna().copy()
    tmp[value_col] = tmp[value_col].astype(str)
    out = (
        tmp.groupby(cluster_col)[value_col]
        .apply(lambda s: s.value_counts().head(top_n))
    )
    return out

top_brand = top_values_per_cluster(df_customer, CL, "favorite_brand", top_n=5)
top_hs    = top_values_per_cluster(df_customer, CL, "favorite_hs", top_n=5)

print("\n=== Top favorite_brand por cluster ===")
if top_brand is not None: print(top_brand)

print("\n=== Top favorite_hs por cluster ===")
if top_hs is not None: print(top_hs)


In [ ]:
OUT_DIR = Path(r"CSVTables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_customer_out = df_customer.copy()

# guardar
df_customer_out.to_csv(OUT_DIR / "DF_CUSTOMER_PERSONAS.csv", index=False, encoding="utf-8")
print("Saved:", OUT_DIR / "DF_CUSTOMER_PERSONAS.csv", "| shape:", df_customer_out.shape)

persona_table.to_csv(OUT_DIR / "PERSONAS.csv", index=False, encoding="utf-8")
print("Saved:", OUT_DIR / "PERSONAS.csv", "| shape:", persona_table.shape)

In [ ]:
persona_table